# 10_Save_And_Load_Model.ipynb

# Section 1 — Introduction

This is the final notebook of your **Week 4 Scikit-Learn** roadmap.

After training a model, you don't want to retrain it every time your application starts. Instead, you **save the trained model** to disk and **load it later** whenever needed.

---

# What is Model Persistence?

**Model Persistence** is the process of saving a trained machine learning model to a file so it can be reused later without retraining.

The saved file contains everything the model has learned, such as:

* Learned weights/coefficients
* Hyperparameters
* Model structure
* Preprocessing steps (if using a Pipeline)

---

# Why Do We Need It?

Imagine you trained a model that took:

```text
Training Time = 30 minutes
```

Without saving:

```text
Open Application

↓

Train Model (30 min)

↓

Predict
```

Every time.

Very inefficient.

---

With a saved model:

```text
Open Application

↓

Load Model (1 second)

↓

Predict
```

Much faster.

---

# Mathematical Intuition

Training a model means learning parameters.

For Linear Regression:

[
\hat{y}=Xw+b
]

Training finds:

```text
w

b
```

These learned parameters are stored in the saved model file.

When you load the model later, those exact values are restored.

No retraining is required.

---

# How Does It Work?

```text
Training Data
      │
      ▼
Train Model
      │
      ▼
Learn Parameters
      │
      ▼
Save Model (.pkl / .joblib)
      │
      ▼
──────────────────────────────
Later...
──────────────────────────────
      │
      ▼
Load Model
      │
      ▼
Predict on New Data
```

---

# What Can Be Saved?

Almost any trained scikit-learn object.

Examples:

| Object                   | Can be Saved? |
| ------------------------ | ------------- |
| `LinearRegression`       | ✅             |
| `LogisticRegression`     | ✅             |
| `RandomForestClassifier` | ✅             |
| `Pipeline`               | ✅             |
| `GridSearchCV`           | ✅             |
| `StandardScaler`         | ✅             |

---

# Why Save the Entire Pipeline?

Suppose you trained:

```text
StandardScaler

↓

LogisticRegression
```

If you save only the model:

```text
LogisticRegression
```

Later you must remember to manually scale new data before prediction.

Easy to make mistakes.

Instead, save the **entire Pipeline**.

```text
Pipeline

↓

StandardScaler

↓

LogisticRegression
```

Now:

```python
pipe.predict(new_data)
```

automatically:

* Scales the data
* Makes predictions

---

# Common File Formats

Two formats are commonly used.

| Format | Library  | Extension |
| ------ | -------- | --------- |
| Pickle | `pickle` | `.pkl`    |
| Joblib | `joblib` | `.joblib` |

---

# Pickle vs Joblib

| Pickle                           | Joblib                                                |
| -------------------------------- | ----------------------------------------------------- |
| Built into Python                | Separate library                                      |
| Good for small objects           | Better for large NumPy arrays and scikit-learn models |
| Slightly slower for large models | Faster for loading/saving large models                |
| General-purpose serialization    | Optimized for machine learning                        |

---

# Which Should You Use?

For scikit-learn models:

✅ **Joblib** is generally recommended because it is optimized for objects containing large NumPy arrays.

Pickle is still perfectly valid and commonly seen.

---

# Advantages

| Advantage            | Explanation                                |
| -------------------- | ------------------------------------------ |
| No retraining        | Save time and computation                  |
| Faster deployment    | Load in seconds                            |
| Easy model sharing   | Send the model file to others              |
| Reproducibility      | Same trained model every time              |
| Works with Pipelines | Preserves preprocessing and model together |

---

# Disadvantages

| Disadvantage          | Explanation                                                                                   |
| --------------------- | --------------------------------------------------------------------------------------------- |
| Version compatibility | Different scikit-learn versions may not load older models correctly                           |
| Security              | Never load pickle/joblib files from untrusted sources because they can execute arbitrary code |
| Large file sizes      | Complex models can produce large files                                                        |

---

# When to Use

Use model saving when:

* Deploying a machine learning model.
* Building desktop or web applications.
* Reusing a trained model later.
* Sharing models within a team.

---

# When NOT to Use

Avoid saving a model when:

* Training is still in progress.
* The model hasn't been validated yet.
* You plan to change preprocessing or hyperparameters immediately.

---

# Applications

| Application      | Example                                                 |
| ---------------- | ------------------------------------------------------- |
| Web API          | Load model when the server starts and serve predictions |
| Desktop App      | Predict house prices without retraining                 |
| Mobile Backend   | Fraud detection service                                 |
| Batch Processing | Daily prediction jobs                                   |
| Research         | Save the best model from experiments                    |

---

# Pickle vs Joblib vs ONNX

| Feature                   | Pickle | Joblib | ONNX                       |
| ------------------------- | ------ | ------ | -------------------------- |
| Saves scikit-learn models | ✅      | ✅      | Limited conversion support |
| Optimized for ML models   | ❌      | ✅      | N/A                        |
| Cross-language support    | ❌      | ❌      | ✅                          |
| Most common in Python     | ✅      | ✅      | Less common for beginners  |

---

# Key Takeaways

* Model persistence means **saving a trained model** for later use.
* The two common formats are **Pickle** (`.pkl`) and **Joblib** (`.joblib`).
* For scikit-learn projects, **Joblib** is usually the preferred choice.
* Save the **entire Pipeline**, not just the estimator, when preprocessing is involved.
* Never load serialized model files from untrusted sources.

---

## ⭐ Connection to Previous Notebooks

```text
Train Model
      │
      ▼
Pipeline
      │
      ▼
Cross Validation
      │
      ▼
GridSearchCV
      │
      ▼
Best Model
      │
      ▼
Save Model
      │
      ▼
Load Model
      │
      ▼
Predict
```


# 10_Save_And_Load_Model.ipynb

# Section 2 — Import & Constructor

Unlike previous notebooks, **Pickle** and **Joblib** don't have machine learning constructors. They are **serialization libraries** used to save and load Python objects.

We'll cover:

* `pickle.dump()`
* `pickle.load()`
* `joblib.dump()`
* `joblib.load()`

These are the four functions you'll use almost all the time.

---

# Imports

## Using Pickle

```python
import pickle
```

---

## Using Joblib

```python
import joblib
```

---

# Pickle Functions

---

## 1. pickle.dump() ⭐⭐⭐⭐⭐

Used to **save** a Python object.

### Syntax

```python
pickle.dump(
    obj,
    file,
    protocol=None
)
```

### Important Parameters

| Parameter  | Description                                      |
| ---------- | ------------------------------------------------ |
| `obj`      | Object to save (model, pipeline, scaler, etc.)   |
| `file`     | File object opened in binary write mode (`wb`)   |
| `protocol` | Pickle format version (usually leave as default) |

---

### Example

```python
with open("model.pkl", "wb") as f:
    pickle.dump(pipe, f)
```

Here

```text
pipe
↓

Serialized

↓

model.pkl
```

---

# 2. pickle.load() ⭐⭐⭐⭐⭐

Loads a saved object.

### Syntax

```python
pickle.load(file)
```

### Parameter

| Parameter | Description                                   |
| --------- | --------------------------------------------- |
| `file`    | File object opened in binary read mode (`rb`) |

---

### Example

```python
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)
```

Now

```python
loaded_model.predict(X_test)
```

works exactly like the original model.

---

# Joblib Functions

---

## 1. joblib.dump() ⭐⭐⭐⭐⭐

Save an object.

### Syntax

```python
joblib.dump(
    value,
    filename,
    compress=0
)
```

### Important Parameters

| Parameter  | Default | Description                  |
| ---------- | ------- | ---------------------------- |
| `value`    | —       | Object to save               |
| `filename` | —       | File name                    |
| `compress` | `0`     | Compression level (optional) |

---

### Example

```python
joblib.dump(
    pipe,
    "model.joblib"
)
```

Creates

```text
model.joblib
```

---

### Compression

```python
joblib.dump(
    pipe,
    "model.joblib",
    compress=3
)
```

Higher compression

* Smaller file
* Slightly slower save/load

Usually

```python
compress=3
```

is a good balance.

---

# 2. joblib.load() ⭐⭐⭐⭐⭐

Loads a saved object.

### Syntax

```python
joblib.load(filename)
```

---

### Example

```python
loaded_model = joblib.load(
    "model.joblib"
)
```

Now

```python
loaded_model.predict(X_test)
```

works immediately.

---

# Which Should You Use?

### Pickle

```python
with open("model.pkl","wb") as f:
    pickle.dump(model,f)
```

### Joblib

```python
joblib.dump(
    model,
    "model.joblib"
)
```

For scikit-learn,

**Joblib is generally preferred**.

---

# Saving Different Objects

You can save almost anything.

### Save Model

```python
joblib.dump(
    model,
    "model.joblib"
)
```

---

### Save Pipeline

```python
joblib.dump(
    pipe,
    "pipeline.joblib"
)
```

---

### Save GridSearchCV

```python
joblib.dump(
    grid,
    "grid.joblib"
)
```

When loaded,

you still have

```python
grid.best_params_

grid.predict()
```

available.

---

### Save Scaler

```python
joblib.dump(
    scaler,
    "scaler.joblib"
)
```

Although in practice,

saving the **Pipeline** is usually better.

---

# Binary Modes

Pickle requires binary mode.

| Mode   | Meaning      |
| ------ | ------------ |
| `"wb"` | Write Binary |
| `"rb"` | Read Binary  |

Example

```python
with open(
    "model.pkl",
    "wb"
) as f:
```

and

```python
with open(
    "model.pkl",
    "rb"
) as f:
```

---

# Recommended Settings

### Pickle

```python
with open(
    "model.pkl",
    "wb"
) as f:

    pickle.dump(
        pipe,
        f
    )
```

---

### Joblib

```python
joblib.dump(
    pipe,
    "pipeline.joblib",
    compress=3
)
```

---

# Constructor Summary

| Function        | Purpose     | Most Used? |
| --------------- | ----------- | ---------- |
| `pickle.dump()` | Save object | ⭐⭐⭐⭐⭐      |
| `pickle.load()` | Load object | ⭐⭐⭐⭐⭐      |
| `joblib.dump()` | Save object | ⭐⭐⭐⭐⭐      |
| `joblib.load()` | Load object | ⭐⭐⭐⭐⭐      |

---

# ⭐ Interview Questions

### Q1. Why is Joblib preferred over Pickle for scikit-learn?

**Answer:**

Because Joblib is optimized for objects containing large NumPy arrays, making it generally faster and more efficient for saving and loading machine learning models.

---

### Q2. Why do we use `"wb"` and `"rb"` with Pickle?

**Answer:**

Pickle stores data in **binary format**. Therefore:

* `"wb"` → Write Binary
* `"rb"` → Read Binary

---

### Q3. Should you save only the model or the entire Pipeline?

**Answer:**

Save the **entire Pipeline** whenever preprocessing is involved. This ensures that preprocessing and prediction are performed consistently when the model is loaded later.

---

# 10_Save_And_Load_Model.ipynb

# Section 3 — Methods & Attributes

Unlike `Pipeline` or `GridSearchCV`, **Pickle** and **Joblib** have very few functions.

You'll mainly use:

* `dump()`
* `load()`

Everything else is rarely needed for scikit-learn.

---

# Pickle Methods

---

# 1. `pickle.dump()` ⭐⭐⭐⭐⭐

Used to save a Python object.

### Syntax

```python
pickle.dump(obj, file)
```

### Example

```python
import pickle

with open("pipeline.pkl", "wb") as f:
    pickle.dump(pipe, f)
```

Returns

```python
None
```

It simply writes the serialized object to the file.

---

# 2. `pickle.load()` ⭐⭐⭐⭐⭐

Loads the saved object.

### Syntax

```python
pickle.load(file)
```

### Example

```python
with open("pipeline.pkl", "rb") as f:
    loaded_pipe = pickle.load(f)
```

Now

```python
loaded_pipe.predict(X_test)
```

works exactly like the original pipeline.

---

# Joblib Methods

---

# 1. `joblib.dump()` ⭐⭐⭐⭐⭐

Saves an object.

### Syntax

```python
joblib.dump(
    value,
    filename,
    compress=0
)
```

### Example

```python
import joblib

joblib.dump(
    pipe,
    "pipeline.joblib"
)
```

Returns

```python
['pipeline.joblib']
```

A list containing the saved filename.

---

# 2. `joblib.load()` ⭐⭐⭐⭐⭐

Loads the object.

### Syntax

```python
joblib.load(filename)
```

### Example

```python
loaded_pipe = joblib.load(
    "pipeline.joblib"
)
```

Now

```python
loaded_pipe.predict(X_test)
```

works immediately.

---

# Important "Attributes"

Neither Pickle nor Joblib have useful attributes.

Instead, after loading the object, **you access the attributes of the loaded object itself**.

For example, if you saved a `GridSearchCV` object:

```python
joblib.dump(
    grid,
    "grid.joblib"
)
```

Later

```python
loaded_grid = joblib.load(
    "grid.joblib"
)
```

You can still use:

```python
loaded_grid.best_params_

loaded_grid.best_score_

loaded_grid.best_estimator_

loaded_grid.cv_results_
```

Because `loaded_grid` is a `GridSearchCV` object.

---

If you saved a Pipeline

```python
loaded_pipe = joblib.load(
    "pipe.joblib"
)
```

You can still use

```python
loaded_pipe.predict(X_test)

loaded_pipe.score(X_test, y_test)

loaded_pipe.named_steps
```

Exactly as before.

---

If you saved Logistic Regression

```python
loaded_model = joblib.load(
    "model.joblib"
)
```

You can access

```python
loaded_model.coef_

loaded_model.intercept_

loaded_model.classes_

loaded_model.predict(X_test)
```

Everything is preserved.

---

# Complete Example

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import joblib

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression())
])

pipe.fit(X_train, y_train)

joblib.dump(
    pipe,
    "wine_pipeline.joblib"
)

loaded_pipe = joblib.load(
    "wine_pipeline.joblib"
)

y_pred = loaded_pipe.predict(X_test)

print(
    loaded_pipe.score(
        X_test,
        y_test
    )
)
```

---

# Which Objects Can Be Saved?

| Object                   | Can Save? |
| ------------------------ | --------- |
| `LinearRegression`       | ✅         |
| `LogisticRegression`     | ✅         |
| `RandomForestClassifier` | ✅         |
| `Pipeline`               | ✅         |
| `GridSearchCV`           | ✅         |
| `StandardScaler`         | ✅         |

---

# Summary

## Methods

| Method          | Purpose     | Returns                |
| --------------- | ----------- | ---------------------- |
| `pickle.dump()` | Save object | `None`                 |
| `pickle.load()` | Load object | Original Python object |
| `joblib.dump()` | Save object | List of filenames      |
| `joblib.load()` | Load object | Original Python object |

---

## Attributes

There are **no important attributes** for Pickle or Joblib themselves.

The loaded object retains **all the methods and attributes** of its original class.

---

# ⭐ Interview Questions

### Q1. Does saving and loading a model change its learned parameters?

**Answer:**

No. The loaded model contains the same learned parameters, hyperparameters, and state as when it was saved.

---

### Q2. If you save a `Pipeline`, do you lose access to its methods?

**Answer:**

No. After loading, the object is still a `Pipeline`, so methods like `predict()`, `score()`, and attributes like `named_steps` work exactly as before.

---

### Q3. Can you save a `GridSearchCV` object?

**Answer:**

Yes. After loading, you can still access attributes like `best_params_`, `best_score_`, `best_estimator_`, and `cv_results_`.

---